**IMPORTS NECESARIOS Y CONFIGURACIÓN GLOBAL**

Para llevar a cabo la detección y lectura de matrículas, inicialmente se importan las librerías necesarias. En esta versión el OCR clásico se sustituye por `PaddleOCR-VL`.

In [1]:
import os
from functools import partial
from pathlib import Path
import utils_matriculas as um


import torch
from ultralytics import YOLO

from utils_matriculas import (
    GT_KEYS,
    cargar_gt,
    evaluar_dataset,
    get_ocr_vlm,
    leer_placa_vlm,
    procesar_video_yolo,
)

**INICIALIZACIÓN GLOBAL DE MODELOS**

Seleccionaremos el detector de matrículas entrenado `best-yolo.pt`.

In [2]:
# Ruta al detector entrenado de placas
WEIGHTS_PLACAS = "weights/best-yolo.pt"

# Inicializacion del detector de matriculas compartido
detector_placas = YOLO(WEIGHTS_PLACAS)

**CARGA DEL MODELO VLM**

La carga se delega en `get_ocr_vlm`, que devuelve el par `(modelo, processor)` del VLM y lo guarda en caché para cargarlo una sola vez. Aquí se fija el modelo `VLM_ID` y el dispositivo `DEVICE`.

In [3]:
# Identificador del modelo VLM y dispositivo de ejecucion
VLM_ID = "PaddlePaddle/PaddleOCR-VL"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Inicializamos el VLM globalmente: devuelve (modelo, processor), o None si falla la carga
ocr_global = get_ocr_vlm(DEVICE, VLM_ID)

c:\Users\enriq\anaconda3\envs\matriculas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


**LECTURA DE MATRÍCULAS CON VLM (`leer_placa`)**

La lectura está implementada en `leer_placa_vlm`: mismo preprocesamiento que en la versión con PaddleOCR y lectura delegada en el VLM mediante una función fija que pide solo los caracteres de la placa.

Como un VLM no devuelve una confianza por lectura, se estima como la **media de la probabilidad de los tokens generados**, de modo que conserva la firma `(texto, confianza)`. Aquí se fijan con `partial` el dispositivo y el modelo ya cargado, para que `leer_placa(crop)` siga recibiendo solo el recorte.

In [ ]:
leer_placa = partial(leer_placa_vlm, device=DEVICE, ocr_inst=ocr_global)

**TAREA 1: ADQUISICIÓN Y EVALUACIÓN DE MÉTRICAS DE LECTURA CON VLM**

Evaluamos el rendimiento del conjunto detector de matrículas + VLM sobre las imágenes anotadas, comparando contra el texto real. Se obtienen:
- **Exact match**: Proporción de matrículas leídas exactamente igual que la anotación.
- **Acierto por carácter**: Similitud media basada en la distancia de Levenshtein.
- **Techo del OCR**: Lectura directa del recorte de la anotación, sin depender del detector.
- **Lectura cruda vs. corregida**: La lectura tal cual y tras la corrección por posición.
- **Tiempos**: Milisegundos por lectura (media y mediana), con una lectura de calentamiento previa.

La función `evaluar_dataset`: es independiente del detector, que se le pasa como una función `detectar(img) -> [(caja_xyxy, confianza)]`.
- **Análisis de falsos positivos**: Se diferencian las detecciones sin coincidencia con ninguna caja real de aquellas que coinciden con una placa anotada pero sin texto legible (`None`/vacía).

**CONFIGURACIÓN DEL DATASET DE EVALUACIÓN**

Rutas, umbrales y carpeta de salida donde se guardan el CSV de resultados y los recortes con error.

In [5]:
# Configuracion de rutas y parametros del dataset
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"
SPLIT = "train"
MAX_IMAGENES = 200     # None para evaluar el dataset completo
CONF_METRICAS = 0.25
IOU_MATCH = 0.5
PAD = 0.05
IMGSZ_METRICAS = 960
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("vlm_eval")

**FUNCIONES AUXILIARES DE EVALUACIÓN**

`cargar_gt` recibe la carpeta de anotaciones y las claves como argumentos, así que aquí se fija con `partial` (`gt_dataset`) para usarla con `ANOTACIONES`.

`detectar_yolo` adapta el detector YOLO a lo que espera `evaluar_dataset`.

In [6]:
# cargar_gt (utils) recibe la carpeta de anotaciones y las claves como argumentos: las fijamos aqui
gt_dataset = partial(cargar_gt, anotaciones_dir=ANOTACIONES, gt_keys=GT_KEYS)


# Adaptador del detector YOLO para evaluar_dataset
def detectar_yolo(img, detector=detector_placas):
    r = detector.predict(img, imgsz=IMGSZ_METRICAS, conf=CONF_METRICAS, verbose=False)[0]
    return [(b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]

**EJECUCIÓN DE LA EVALUACIÓN**

Se evalúa un subconjunto de imágenes del dataset `MAX_IMAGENES` con la función única `evaluar_dataset`, que da la lectura cruda y la corregida por posición, los tiempos y el techo del OCR.

In [ ]:
# Ejecucion de la evaluacion de metricas
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista, detectar_yolo, leer_placa, gt_dataset, SALIDA, "errores-yolo",
                    "yolo_resultados_vlm.csv", "YOLO", iou_match=IOU_MATCH, pad=PAD,
                    gt_keys=GT_KEYS, delimiter=";")
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset (variable BASE).")

**TAREA 2: DETECCIÓN, TRACKING Y LECTURA DE MATRÍCULAS EN VÍDEO (VLM)**

En este módulo se procesa una secuencia de vídeo realizando:
- **Seguimiento con ByteTrack**: Detección y seguimiento continuo de personas y vehículos (`persona`, `coche`, `moto`, `guagua`, `camion`).
- **Filtrado y validación sintáctica**: Verificación de las matrículas leídas mediante expresiones regulares de formatos de placas europeos.
- **Fusión de tracklets**: Recuperación de identificadores ante oclusiones temporales, tanto para vehículos en movimiento como para vehículos estacionados.
- **Renderizado y exportación**: Generación del vídeo anotado con los identificadores y recuadros de matrícula, junto con el archivo `resultado.csv` detallando las detecciones por fotograma.

**CONFIGURACIÓN DE RUTAS, CLASES Y PALETA**

Rutas del vídeo de entrada y de salida (`resultado.mp4` y `resultado.csv` se guardan en la carpeta `YOLO_VLM_result`), mapeo de clases COCO y colores de dibujo.

In [7]:
# Configuracion de rutas de video
filename = "C0142.MP4"
# Carpeta de resultados: se crea en el directorio de trabajo del cuaderno
carpeta = os.path.abspath("YOLO_VLM_result")
os.makedirs(carpeta, exist_ok=True)
output_path = os.path.join(carpeta, "resultado.mp4")
csv_path = os.path.join(carpeta, "resultado.csv")

# Mapeo de clases y paleta visual
nombre = {0: "persona", 2: "coche", 3: "moto", 5: "guagua", 7: "camion"}
colores = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255),
           5: (0, 255, 255), 7: (255, 255, 0)}

COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35
CONF_PLACA = 0.4
MIN_FRAMES_PLACA = 5

**PARÁMETROS DE LECTURA**

Control de cuándo se lanza la lectura sobre cada vehículo. Los patrones sintácticos de matrículas europeas `PATRONES_MATRICULA` y la función `matricula_valida`.

In [8]:
# Parametros del OCR sobre video
OCR_CADA = 10
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 3

**PARÁMETROS DE TRACKING**

Parámetros del voto de clase y configuración del archivo de ByteTrack.

In [9]:
# Parametros de tracking
MIN_FRAMES = 10
PESO_BORDE = 0.3
MARGEN_BORDE = 5

# Configuracion del archivo de ByteTrack
os.makedirs("./YOLO", exist_ok=True)
with open("./YOLO/mi_bytetrack.yaml", "w") as f:
    f.write("tracker_type: bytetrack\ntrack_high_thresh: 0.25\ntrack_low_thresh: 0.1\n"
            "new_track_thresh: 0.25\ntrack_buffer: 140\nmatch_thresh: 0.8\nfuse_score: True\n")

Finalmente, `procesar_video_yolo` (en `utils_matriculas.py`) integra las dos pasadas:
1. **Pasada 1**: Ejecución de inferencia sobre el vídeo, seguimiento con ByteTrack, detección de matrículas mediante `detector_placas`, lectura con el VLM mediante `leer_placa` y fusión temporal de identidades.
2. **Pasada 2**: Renderizado del vídeo final `resultado.mp4` superponiendo las cajas y matrículas leídas, y volcado a `resultado.csv`.

**EJECUCIÓN DEL VÍDEO**

In [ ]:
if os.path.exists(filename):
    procesar_video_yolo(
        filename, output_path, csv_path, detector_placas, leer_placa, nombre, colores,
        conf_placa=CONF_PLACA, min_frames_placa=MIN_FRAMES_PLACA, color_placa=COLOR_PLACA,
        escala_texto_placa=ESCALA_TEXTO_PLACA, ocr_cada=OCR_CADA, min_ancho_ocr=MIN_ANCHO_OCR,
        min_chars_placa=MIN_CHARS_PLACA, ocr_score_ok=OCR_SCORE_OK,
        ocr_max_intentos=OCR_MAX_INTENTOS, min_frames=MIN_FRAMES,
        peso_borde=PESO_BORDE, margen_borde=MARGEN_BORDE,
        track_kwargs=dict(quantize=16, verbose=False),
        predict_kwargs=dict(quantize=16, verbose=False),
        etiqueta_ocr="VLM", ocr_solo_en_placas=True,
        max_ancho_salida=1920, mostrar_progreso=True,
    )
else:
    print(f"Vídeo {filename} no encontrado. Comprueba la ruta.")

In [13]:
CONF_METRICAS = 0.25
IMGSZ_METRICAS = 960

# Configurar OCR-VL
leer_placa = partial(
    um.leer_placa_vlm,
    device=DEVICE,
    ocr_inst=ocr_global
)

# Adaptador del detector YOLO
def detectar_yolo(img):
    resultados = detector_placas.predict(
        source=img,
        conf=CONF_METRICAS,
        imgsz=IMGSZ_METRICAS,
        verbose=False
    )

    predicciones = []

    for resultado in resultados:
        for caja in resultado.boxes:
            x1, y1, x2, y2 = caja.xyxy[0].cpu().numpy()
            conf = float(caja.conf[0])

            predicciones.append((
                [float(x1), float(y1), float(x2), float(y2)],
                conf
            ))

    return predicciones


# Ejecutar evaluación
resultados_yolo = um.evaluar_detector(
    detector=detectar_yolo,
    imagenes_dir=BASE / "images",
    anotaciones_dir=BASE / "annotations",
    salida_dir=Path("resultados") / "YOLO_VLM",
    leer_ocr=leer_placa,
    nombre_modelo="YOLO_VLM",
    max_imagenes=200,
    iou_match=0.5,
    pad=0.05,
    limpiar_salida=True
)


Modelo: YOLO_VLM
Imágenes: 200
[1/200] 00000.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[2/200] 00001.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[3/200] 00002.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[4/200] 00003.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[5/200] 0173FVB-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[6/200] 0224CXR-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[7/200] 0270DRN.JPG | GT: 2 | Detectadas: 2 | Fallos: 0
[8/200] 0404HCF-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[9/200] 0617KFZ.JPG | GT: 2 | Detectadas: 2 | Fallos: 0
[10/200] 0696CMB.JPG | GT: 2 | Detectadas: 2 | Fallos: 0
[11/200] 0848GLN-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[12/200] 09d2d922-35f1-4ed3-93a5-82a1ed54d8cb.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[13/200] 1 (1).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[14/200] 1 (10).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[15/200] 1 (11).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[16/200] 1 (12).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[17/200] 1 (13).jpeg | GT: 1 